In [1]:
import numpy as np
import pandas as pd
import spacy
from spacy.lang.en import English
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.base import TransformerMixin
from sklearn.pipeline import Pipeline

In [2]:
nlp = English()

In [3]:
train = pd.read_csv('/kaggle/input/nlp-getting-started/train.csv', index_col = 'id')
test = pd.read_csv('/kaggle/input/nlp-getting-started/test.csv', index_col = 'id')
submit = pd.read_csv('/kaggle/input/nlp-getting-started/sample_submission.csv', index_col = 'id')
train.head()

,keyword,location,text,target
id,,,,
1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
5,NaN,NaN,All residents asked to 'shelter in place' are ...,1
6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1
7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1


In [4]:
train.target.value_counts()

0    4342
1    3271
Name: target, dtype: int64

In [5]:
import string
from spacy.lang.en.stop_words import STOP_WORDS
from spacy.lang.en import English

punctuations = string.punctuation
nlp = spacy.load('en')
stop_words = spacy.lang.en.stop_words.STOP_WORDS

parser = English()

def spacy_tokenizer(sentence):
    mytokens = parser(sentence)
    mytokens = [ word.lemma_.lower().strip() if word.lemma_ != "-PRON-" else word.lower_ for word in mytokens ]
    mytokens = [ word for word in mytokens if word not in stop_words and word not in punctuations ]
    return mytokens


In [6]:
class predictors(TransformerMixin):
    def transform(self, X, **transform_params):
        return [clean_text(text) for text in X]

    def fit(self, X, y=None, **fit_params):
        return self

    def get_params(self, deep=True):
        return {}

def clean_text(text):
    return text.strip().lower()

In [7]:
bow_vector = CountVectorizer(tokenizer = spacy_tokenizer, ngram_range=(1,1))

In [8]:
tfidf_vector = TfidfVectorizer(tokenizer = spacy_tokenizer, ngram_range= (1, 1))

In [9]:
from sklearn.model_selection import train_test_split
X = train.text
y = train.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3)

In [10]:
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression()

pipe = Pipeline([
    ("cleaner" , predictors()),
    ("vectorizer" , bow_vector),
    ("predictor" , clf)])
pipe.fit(X_train,y_train)

/opt/conda/lib/python3.6/site-packages/sklearn/linear_model/logistic.py:432: FutureWarning: Default solver will be changed to 'lbfgs' in 0.22. Specify a solver to silence this warning.
  FutureWarning)


Pipeline(memory=None,
         steps=[('cleaner', <__main__.predictors object at 0x7f2496bc5278>),
                ('vectorizer',
                 CountVectorizer(analyzer='word', binary=False,
                                 decode_error='strict',
                                 dtype=<class 'numpy.int64'>, encoding='utf-8',
                                 input='content', lowercase=True, max_df=1.0,
                                 max_features=None, min_df=1,
                                 ngram_range=(1, 1), preprocessor=None,
                                 stop_words=None, strip_accents=None,
                                 t...)\\b\\w\\w+\\b',
                                 tokenizer=<function spacy_tokenizer at 0x7f2498efa048>,
                                 vocabulary=None)),
                ('predictor',
                 LogisticRegression(C=1.0, class_weight=None, dual=False,
                                    fit_intercept=True, intercept_scaling=1,
            

In [11]:
from sklearn import metrics
predicted = pipe.predict(X_test)

In [12]:
print("accuracy :",  metrics.accuracy_score(predicted, y_test))

accuracy : 0.7889667250437828


In [13]:
submit.head()

,target
id,
0,0
2,0
3,0
9,0
11,0


In [14]:
test_text = test.text
submit.target = pipe.predict(test_text)
submit.head()

,target
id,
0,1
2,1
3,1
9,0
11,1


In [15]:
submit.to_csv('submit.csv')